In [42]:
import os
import pandas as pd
import polars as pl
import yaml
import random
from faker import Faker
import mimesis
fake = Faker('pt_BR')

def load_config(file_path: str):
    """Carrega arquivo YAML"""
    with open(file_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)

In [2]:
# Configura o diretório de trabalho para a raiz do projeto
import os
import sys

# Define o caminho da raiz do projeto diretamente
project_root = '/home/nicolas/Projects/Personal/corporate-benefits-transactions'
os.chdir(project_root)
print(f"Diretório de trabalho atual: {os.getcwd()}")

# Adiciona o diretório raiz ao path do Python para imports
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Carrega configurações usando os novos arquivos reorganizados
business_data = load_config('./config/business_rules.yaml')       # Dados estáticos para DataFrames
parameters = load_config('./config/transaction_simulation.yaml')  # Parâmetros de configuração
validation_rules = load_config('./config/validation_rules.yaml')

print("Configurações carregadas com sucesso!")
print(f"Dados de negócio carregados: {list(business_data.keys())}")
print(f"Parâmetros de simulação carregados: {list(parameters.keys())}")

Diretório de trabalho atual: /home/nicolas/Projects/Personal/corporate-benefits-transactions
Configurações carregadas com sucesso!
Dados de negócio carregados: ['products', 'transaction_modalities', 'acquirers', 'terminal_models_by_acquirer', 'segments', 'capture_technologies', 'status_options', 'cnae_products_mapping']
Parâmetros de simulação carregados: ['data_volumes', 'approval_rates', 'generation_settings', 'usage_patterns']


### 1. Municipalities

In [3]:
# Recarregando o módulo para garantir que temos a versão corrigida
import importlib
import src.extractors.municipality_data_importer
importlib.reload(src.extractors.municipality_data_importer)
from src.extractors.municipality_data_importer import generate_municipality_data

shapefile_path = './data/raw/municipios/municipios.shp'

### 2. Products

In [4]:
def generate_products(config_files_updated):
    """Generates products DataFrame based on business rules"""
    produtos = []
    for codigo, dados in config_files_updated['products'].items():
        produto = {
            'product_id': f"PROD_{dados['code'].upper()}",
            'code': dados['code'],
            'name': dados['name'],
            'category': dados['category'],
            'description': f"Produto de {dados['name']}",
            'active': True,
            'created_at': pd.Timestamp.now()
        }
        produtos.append(produto)

    produtos_df = pd.DataFrame(produtos)
    produtos_df['code'] = produtos_df['code'].astype('category')
    produtos_df['name'] = produtos_df['name'].astype('category')
    produtos_df['category'] = produtos_df['category'].astype('category')

    return produtos_df

### 3. CNAE

In [5]:
from src.extractors.kaggle_downloader import download_kaggle_dataset

def extract_cnae_data():
    download_kaggle_dataset("diovanimerlo/cnae-brazil-2024")

    cnaes = pd.read_csv(
        './data/raw/cnae_dataset_2024.csv',
        sep=',',
        encoding='utf-8',
        dtype={
            "secao_id": str,
            "secao": "category",
            "divisao_id": str,
            "divisao": "category",
            "grupo_id": str,
            "grupo": "category",
            "classe_id": str,
            "classe": "category",
            "subclasse_id": str,
            "subclasse": "category"
        }
    )
    return cnaes

### 4. Segments

In [6]:
def generate_segments(config_files_updated):
    """Generate segments DataFrame from business rules"""
    segments_list = config_files_updated['segments']
    segments = pd.DataFrame(segments_list)
    segments['created_at'] = pd.Timestamp.now()
    segments['name'] = segments['name'].astype('category')
    segments['category'] = segments['category'].astype('category')
    return segments


### 5. Acquirers

In [7]:
def generate_acquirers(config_files_updated):
    """Generates DataFrame for Acquirers based on business rules"""
    acquirers = []
    for idx, name in enumerate(config_files_updated['acquirers'], start=1):
        acquirer = {
            "acquirer_id": f"ACQ_{idx:03d}",
            "name": name,
            "active": True,
            "created_at": pd.Timestamp.now()
        }
        acquirers.append(acquirer)
    return pd.DataFrame(acquirers)

### 6. Transaction Modalities

In [8]:
def generate_transaction_modalities(config_files_updated):
    """Generates DataFrame for Transaction Modalities based on business rules"""
    transaction_modalities = []
    modalidades_yaml = config_files_updated['transaction_modalities']

    # Se for dict, usa .values(); se for lista, itera direto
    iterator = modalidades_yaml.values() if isinstance(modalidades_yaml, dict) else modalidades_yaml

    for idx, dados in enumerate(iterator, start=1):
        modality = {
            "modality_id": f"MOD_{idx:03d}",
            "code": dados["name"].upper().replace(" ", "_"),
            "name": dados["name"],
            "type": "brand" if dados["name"].upper() in ["ELO", "MASTERCARD", "VISA"] else "proprietary_network",
            "active": True,
            "created_at": pd.Timestamp.now()
        }
        transaction_modalities.append(modality)
    return pd.DataFrame(transaction_modalities)

### 7. MCC Categories

In [9]:
import pandas as pd

def extract_mcc_data():
    mcc = pd.read_csv(
        './data/raw/mcc.csv',
        sep=',',
        encoding='utf-8',
        dtype={
            "mcc_id": str,
            "mcc_code": str,
            "mcc_description": str,
            "valid_payment_brands": str,
            "category": "category"
        }
    )
    return mcc

### 8. Capture Technologies

In [10]:
def generate_capture_technologies():
    """Generate capture technologies"""
    technologies = [
        {"code": "CHIP", "category": "physical", "name": "Chip", "requires_physical_card": True},
        {"code": "MAG_STRIPE", "category": "physical", "name": "Magnetic Stripe", "requires_physical_card": True},
        {"code": "CONTACTLESS", "category": "physical", "name": "Contactless", "requires_physical_card": True},
        {"code": "MOBILE_NFC", "category": "digital", "name": "Mobile NFC", "requires_physical_card": False},
        {"code": "APPLE_PAY", "category": "digital", "name": "Apple Pay", "requires_physical_card": False},
        {"code": "GOOGLE_PAY", "category": "digital", "name": "Google Pay", "requires_physical_card": False}
    ]

    capture_tech_list = []
    for idx, tech in enumerate(technologies, start=1):
        capture_tech = {
            "technology_id": f"TECH_{idx:03d}",
            "technology_code": tech["code"],
            "technology_category": tech["category"],
            "technology_name": tech["name"],
            "requires_physical_card": tech["requires_physical_card"],
            "created_at": pd.Timestamp.now()
        }
        capture_tech_list.append(capture_tech)

    return pd.DataFrame(capture_tech_list)

### 9. Establishments 

In [44]:
def generate_establishments(n_establishments, df_cnae, df_mcc, df_municipalities, cnae_with_products_ratio=0.85):
    """Generate establishments DataFrame with intelligent CNAE selection
    
    Args:
        n_establishments: Number of establishments to generate
        df_cnae: DataFrame with CNAE codes
        df_mcc: DataFrame with MCC codes
        df_municipalities: DataFrame with municipalities
        cnae_with_products_ratio: Ratio of establishments that should have CNAEs accepting products (default 0.85)
    """
    establishments = []
    
    # Load business rules to get CNAEs that accept products
    business_data = business_data = load_config('./config/business_rules.yaml') 
    cnae_products_mapping = business_data.get('cnae_products_mapping', {})
    
    # Get CNAEs that accept products and all available CNAEs
    cnaes_with_products = set()
    for cnae_pattern in cnae_products_mapping.keys():
        # Find CNAEs that match patterns (exact or partial)
        matching_cnaes = df_cnae[df_cnae['subclasse_id'].str.startswith(cnae_pattern)]['subclasse_id'].tolist()
        cnaes_with_products.update(matching_cnaes)
    
    cnaes_with_products = list(cnaes_with_products)
    all_cnaes = df_cnae['subclasse_id'].tolist()
    cnaes_without_products = [cnae for cnae in all_cnaes if cnae not in cnaes_with_products]
    
    # Calculate how many establishments should have product-accepting CNAEs
    n_with_products = int(n_establishments * cnae_with_products_ratio)
    n_without_products = n_establishments - n_with_products
    
    print(f"Generating {n_establishments} establishments:")
    print(f"  - {n_with_products} with CNAEs that accept products ({cnae_with_products_ratio:.1%})")
    print(f"  - {n_without_products} with random CNAEs ({1-cnae_with_products_ratio:.1%})")
    print(f"  - {len(cnaes_with_products)} CNAEs accept products out of {len(all_cnaes)} total CNAEs")
    
    # Tipos de natureza jurídica mais comuns
    legal_natures = ["LTDA", "EIRELI", "SA", "MEI", "EPP", "COOPERATIVA"]
    company_sizes = ["MEI", "ME", "EPP", "PEQUENO PORTE", "MEDIO PORTE", "GRANDE PORTE"]
    statuses = ["ATIVA", "INATIVA", "SUSPENSA", "BAIXADA"]
    status_reasons = ["NORMAL", "IRREGULARIDADE", "FALTA DE MOVIMENTO", "MUDANÇA DE ENDEREÇO"]
    
    for i in range(n_establishments):
        # Intelligent CNAE selection
        if i < n_with_products and cnaes_with_products:
            # Select from CNAEs that accept products
            main_cnae_id = random.choice(cnaes_with_products)
        else:
            # Select from any CNAE (including those without products)
            main_cnae_id = random.choice(all_cnaes)
        
        establishment = {
            'establishment_id': f"EST_{i:06d}",
            'cnpj': fake.cnpj(),
            'legal_name': fake.company(),
            'trade_name': fake.company_suffix() + " " + fake.last_name(),
            'main_cnae_id': main_cnae_id,
            'mcc_code': random.choice(df_mcc['mcc_code'].tolist()),
            'legal_nature': random.choice(legal_natures),
            'company_size': random.choice(company_sizes),
            'municipality_id': random.choice(df_municipalities['municipality_id'].tolist()),
            'postal_code': fake.postcode(),
            'street': fake.street_name(),
            'number': str(fake.building_number()),
            'complement': random.choice([None, "Sala " + str(random.randint(1, 100)), "Loja " + str(random.randint(1, 50))]),
            'neighborhood': fake.neighborhood(),
            'email': fake.company_email(),
            'phone': fake.phone_number(),
            'registration_status': random.choice(statuses),
            'status_date': fake.date_between(start_date='-2y', end_date='today'),
            'status_reason': random.choice(status_reasons),
            'opening_date': fake.date_between(start_date='-10y', end_date='-1y'),
            'created_at': pd.Timestamp.now()
        }
        establishments.append(establishment)
    
    return pd.DataFrame(establishments)

### 10. Companies

In [12]:
def generate_companies(n_companies, municipalities, segments):
    companies = []
    for i in range(n_companies):
        company = {
            'company_id': f"EMP_{i}",
            'parent_company_id': random.choice([None] + [c["company_id"] for c in companies]) if companies else None,
            'name': fake.company(),
            'cnpj': fake.cnpj(),
            'company_type': random.choice(["Privada", "Publica"]),
            'segment_id': random.choice(segments['segment_id'].tolist()),
            'municipality_id': random.choice(municipalities['municipality_id'].tolist()),
            'contract_registration_date': fake.date_between(start_date='-5y', end_date='today'),
            'created_at': pd.Timestamp.now()
        }
        companies.append(company)
    return pd.DataFrame(companies)  # Return as DataFrame for consistency

### 11. Cardholders

In [13]:
def generate_cardholders(n_funcionarios, companies_df, municipalities_df):
    """Generate cardholders DataFrame based on existing companies data"""
    cardholders = []
    
    for i in range(n_funcionarios):
        cardholder = {
            'cardholder_id': f"FUNC_{i}",
            'company_id': random.choice(companies_df['company_id'].tolist()),
            'name': fake.name(),
            'cpf': fake.cpf(),
            'municipality_id': random.choice(municipalities_df['municipality_id'].tolist()),
            'admission_date': fake.date_this_decade(),
            'position': fake.job(),
            'department': fake.word(),
            'status': random.choice(['active', 'inactive', 'terminated']),
            'created_at': pd.Timestamp.now()
        }
        cardholders.append(cardholder)
    
    return pd.DataFrame(cardholders)

### 12. Cards

In [14]:
def generate_cards(n_cards, cardholders_df, modality_df):
    cards = []

    for idx in range(n_cards):
        card_number = "".join([str(random.randint(0, 9)) for _ in range(16)])
        formatted_card_number = " ".join([card_number[i:i+4] for i in range(0, 16, 4)])
        
        cvv = "".join([str(random.randint(0, 9)) for _ in range(3)])

        card = {
            "card_id": idx,
            "cardholder_id": random.choice(cardholders_df['cardholder_id'].tolist()),
            "card_number": formatted_card_number,
            "modality_id": random.choice(modality_df['modality_id'].tolist()),
            "expiration_date": fake.date_between(start_date='today', end_date='+5y'),
            "cvv": cvv,
            "card_type": random.choice(['credit', 'debit']),
            "issuance_date": pd.Timestamp.now(),
            "status": random.choice(['active', 'inactive', 'blocked', 'canceled']),
            "created_at": pd.Timestamp.now()
        }
        cards.append(card)

    return pd.DataFrame(cards)

### 13. Card Products (TODO)

In [15]:
def generate_card_products(cards_df, products_df, min_products=1, max_products=3):
    """
    Gera produtos associados aos cartões.
    Cada cartão pode ter entre min_products e max_products produtos.
    """
    card_products = []

    for card_id in cards_df['card_id'].tolist():
        n_products = random.randint(min_products, max_products)

        for _ in range(n_products):
            activation_date = fake.date_between(start_date='-2y', end_date='today')

            # 30% de chance de já ter sido desativado
            if random.random() < 0.3:
                deactivation_date = fake.date_between(start_date=activation_date, end_date='today')
                status = random.choice(['suspended', 'cancelled'])
            else:
                deactivation_date = None
                status = 'active'

            card_product = {
                "card_product_id": fake.uuid4(),
                "card_id": card_id,
                "product_id": random.choice(products_df['product_id'].tolist()),
                "current_balance": round(random.uniform(0, 5000), 2),
                "daily_limit": round(random.uniform(100, 2000), 2),
                "monthly_limit": round(random.uniform(1000, 10000), 2),
                "activation_date": activation_date,
                "deactivation_date": deactivation_date,
                "status": status,
                "created_at": pd.Timestamp.now()
            }
            card_products.append(card_product)

    return pd.DataFrame(card_products)


### 14.Terminals

In [16]:
def generate_terminals(n_terminals, establishments_df, acquirers_df, business_data):
    """Generate terminals for establishments with correct model/acquirer/type relation"""
    terminals = []

    models_by_acquirer = business_data["terminal_models_by_acquirer"]
    # Mapeamento modelo -> tipo
    model_type_map = {
        "POS Cielo LIO": "POS",
        "TEF Rede": "TEF",
        "Stone Ton": "POS",
        "GetNet POS": "POS",
        "Moderninha Pro": "POS",
        "SafraPay POS": "POS",
        "POS Genérico": "POS"
    }

    for i in range(n_terminals):
        acquirer_name = random.choice(acquirers_df['name'].tolist())
        terminal_model = random.choice(models_by_acquirer.get(acquirer_name, ["POS Genérico"]))
        terminal_type = model_type_map.get(terminal_model, "POS")
        terminal = {
            'terminal_id': f"TERM_{i:08d}",
            'terminal_code': f"T{random.randint(100000, 999999)}",
            'establishment_id': random.choice(establishments_df['establishment_id'].tolist()),
            'acquirer_id': acquirers_df[acquirers_df['name'] == acquirer_name]['acquirer_id'].iloc[0],
            'terminal_model': terminal_model,
            'terminal_type': terminal_type,
            'status': random.choice(['active', 'inactive', 'maintenance']),
            'installation_date': fake.date_between(start_date='-3y', end_date='today'),
            'created_at': pd.Timestamp.now()
        }
        terminals.append(terminal)

    return pd.DataFrame(terminals)

### 15. Generate Response Codes

In [17]:
def generate_response_codes():
    """Generate response codes for transaction processing"""
    response_codes = [
        {"code": "00", "description": "Approved", "category": "success"},
        {"code": "51", "description": "Insufficient funds", "category": "balance"},
        {"code": "14", "description": "Invalid card number", "category": "card"},
        {"code": "55", "description": "Incorrect PIN", "category": "password"},
        {"code": "96", "description": "System error", "category": "network"},
        {"code": "30", "description": "Format error", "category": "network"},
        {"code": "12", "description": "Invalid transaction", "category": "establishment"},
        {"code": "43", "description": "Stolen card", "category": "card"}
    ]

    codes_list = []
    for idx, code_info in enumerate(response_codes, start=1):
        response_code = {
            "response_code_id": f"RESP_{idx:03d}",
            "response_code": code_info["code"],
            "response_description": code_info["description"],
            "response_category": code_info["category"],
            "created_at": pd.Timestamp.now()
        }
        codes_list.append(response_code)

    return pd.DataFrame(codes_list)


###  DataFrames

In [18]:
municipalities_df = generate_municipality_data(shapefile_path)
products_df = generate_products(business_data)
cnaes_df = extract_cnae_data()
segments_df = generate_segments(business_data)
acquirers_df = generate_acquirers(business_data)
transaction_modalities_df = generate_transaction_modalities(business_data)
mcc_df = extract_mcc_data()
capture_tech_df = generate_capture_technologies()
establishments_df = generate_establishments(
    parameters['data_volumes']['establishments'], 
    cnaes_df, 
    mcc_df, 
    municipalities_df,
    parameters['generation_settings'].get('cnae_with_products_ratio', 0.85)
)  # <- sem vírgula
terminals_df = generate_terminals(parameters['data_volumes']['terminals'], establishments_df, acquirers_df, business_data)      # <- sem vírgula
companies_df = generate_companies(parameters['data_volumes']['companies'], municipalities_df, segments_df)
cardholders_df = generate_cardholders(parameters['data_volumes']['cardholders'], companies_df, municipalities_df)
cards_df = generate_cards(parameters['data_volumes']['cards'], cardholders_df, transaction_modalities_df)
card_products_df = generate_card_products(cards_df, products_df, min_products=1, max_products=3)

Path to dataset files: /home/nicolas/.cache/kagglehub/datasets/diovanimerlo/cnae-brazil-2024/versions/1
Arquivo já existe, pulando: /home/nicolas/Projects/Personal/corporate-benefits-transactions/src/extractors/../../data/raw/cnae_dataset_2024.csv
Generating 150 establishments:
  - 127 with CNAEs that accept products (85.0%)
  - 23 with random CNAEs (15.0%)
  - 61 CNAEs accept products out of 1332 total CNAEs


In [19]:
import numpy as np

def gerar_valor_transacao(max_valor=1000, escala=250):
    """
    Gera um valor aleatório progressivo para transação.
    Quanto maior o valor, menor a chance de ocorrer.
    """
    valor = np.random.exponential(scale=escala)
    return round(min(valor, max_valor), 2)

## OLTP process

## 📊 PROCESSO OLTP - Transformação para Estrutura OLTP

Agora vamos transformar os dados gerados para seguir a estrutura das tabelas OLTP, sem ainda inserir no banco.

### Gera Transações

In [36]:
def generate_single_transaction(cards_df, establishments_df, terminals_df, card_products_df):
    """
    Gera uma única transação para simular operação OLTP unitária
    Usa os dados já criados (cards, establishments, etc.)
    
    Returns:
        dict: Uma única transação
    """
    # Só estabelecimentos que têm pelo menos um terminal
    valid_establishments = terminals_df['establishment_id'].unique()
    
    # Selecionar dados aleatórios para a transação
    card_id = cards_df.sample(1)['card_id'].iloc[0]
    establishment_id = random.choice(valid_establishments)
    terminals_for_est = terminals_df[terminals_df['establishment_id'] == establishment_id]
    terminal_id = terminals_for_est['terminal_id'].sample(1).iloc[0]
    product_id = card_products_df[card_products_df['card_id'] == card_id]['product_id'].sample(1).iloc[0]
    amount = gerar_valor_transacao(max_valor=500, escala=100)

    transaction = {
        "transaction_id": str(fake.uuid4()),
        "card_id": card_id,
        "product_id": product_id,
        "establishment_id": establishment_id,
        "terminal_id": terminal_id,
        "amount": amount,
        "date": fake.date_time_between(start_date='-30d', end_date='now'),
        "capture_type": random.choice(['chip', 'magnetic_stripe', 'contactless', 'nfc']),
        "response_code": None,
        "response_message": None,
        "created_at": pd.Timestamp.now(),
        "updated_at": pd.Timestamp.now()
    }
    
    return transaction

# Exemplo de uso: gerar uma única transação
single_transaction = generate_single_transaction(cards_df, establishments_df, terminals_df, card_products_df)
print("Transação gerada:")
print(single_transaction)

Transação gerada:
{'transaction_id': '3c784822-a253-46dd-bdb6-65cc2c52c4a5', 'card_id': np.int64(586), 'product_id': 'PROD_VA', 'establishment_id': 'EST_000076', 'terminal_id': 'TERM_00000171', 'amount': 139.34, 'date': datetime.datetime(2025, 8, 29, 23, 18, 13, 54670), 'capture_type': 'nfc', 'response_code': None, 'response_message': None, 'created_at': Timestamp('2025-09-21 17:22:01.506859'), 'updated_at': Timestamp('2025-09-21 17:22:01.506872')}


## VALIDAÇÕES DE STATUS E ATIVAÇÃO

validation_order:
  - "status_validations"
  - "temporal_validations" 
  - "business_rules_validations"
  - "balance_and_limits_validations"
  - "infrastructure_validations"

###  Status Validations

In [21]:
def validate_card_status(single_transaction, cards_df):
    """
    Valida o status do cartão de uma transação
    
    Args:
        single_transaction: Linha da transação com 'card_id'
        cards_df: DataFrame dos cartões
    
    Returns:
        dict: {'passed': bool, 'error_code': str or None}
    """
    # Códigos de erro por status
    error_codes = validation_rules['status_validations']['card_status']['error_codes']
    
    card_id = single_transaction['card_id']
    
    # Buscar cartão
    card = cards_df[cards_df['card_id'] == card_id]
    
    if card.empty:
        return {'passed': False, 'error_code': '99'}  # Cartão não encontrado
    
    status = card['status'].iloc[0]
    
    # Só passa se for 'active'
    if status == 'active':
        return {'passed': True, 'error_code': None}
    else:
        return {'passed': False, 'error_code': error_codes.get(status, '99')}
    
test_validation_card_status = validate_card_status(single_transaction, cards_df)
print(test_validation_card_status)

{'passed': False, 'error_code': '14'}


In [22]:
def cardholder_status_validation(single_transaction, cards_df, cardholders_df):
  """
  Valida o status do portador do cartão de uma transação

  Args:
      single_transaction: Linha da transação com 'card_id'
      cards_df: DataFrame dos cartões
      cardholders_df: DataFrame dos portadores de cartão

  Returns:
      dict: {'passed': bool, 'error_code': str or None}
  """

  # Códigos de erro por status
  error_codes = {
      'inactive': '16',
      'terminated': '17'
  }

  card_id = single_transaction['card_id']

  # Buscar cartão
  card = cards_df[cards_df['card_id'] == card_id]

  if card.empty:
      return {'passed': False, 'error_code': '99'}  # Cartão não encontrado

  cardholder_id = card['cardholder_id'].iloc[0]

  # Buscar portador do cartão
  cardholder = cardholders_df[cardholders_df['cardholder_id'] == cardholder_id]

  if cardholder.empty:
      return {'passed': False, 'error_code': '98'}  # Portador do cartão não encontrado

  status = cardholder['status'].iloc[0]

  # Só passa se for 'active'
  if status == 'active':
      return {'passed': True, 'error_code': None}
  else:
      return {'passed': False, 'error_code': error_codes.get(status, '99')}
  

In [23]:
def validate_establishment_status(single_transaction, establishments_df):
    """
    Valida o status do estabelecimento de uma transação
    
    Args:
        single_transaction: Linha da transação com 'establishment_id'
        establishments_df: DataFrame dos estabelecimentos
    
    Returns:
        dict: {'passed': bool, 'error_code': str or None}
    """
    # Códigos de erro por status
    error_codes = {
        'INATIVA': '18',
        'SUSPENSA': '19',
        'BAIXADA': '20'
    }
    
    establishment_id = single_transaction['establishment_id']
    
    # Buscar estabelecimento
    establishment = establishments_df[establishments_df['establishment_id'] == establishment_id]
    
    if establishment.empty:
        return {'passed': False, 'error_code': '97'}  # Estabelecimento não encontrado
    
    status = establishment['registration_status'].iloc[0]
    
    # Só passa se for 'ATIVA'
    if status == 'ATIVA':
        return {'passed': True, 'error_code': None}
    else:
        return {'passed': False, 'error_code': error_codes.get(status, '99')}

In [24]:
def validate_terminal_status(single_transaction, terminals_df):
    """
    Valida o status do terminal
    """
    # Forma correta de acessar os códigos de erro do terminal
    terminal_error_codes = validation_rules['status_validations']['terminal_status']['error_codes']
    
    terminal_id = single_transaction['terminal_id']
    
    # Buscar terminal (corrigido o filtro)
    terminal = terminals_df[terminals_df['terminal_id'] == terminal_id]
    
    if terminal.empty:
        return {'passed': False, 'error_code': '96'}  # Terminal não encontrado
    
    status = terminal['status'].iloc[0]
    
    # Só passa se for 'active'
    if status == 'active':
        return {'passed': True, 'error_code': None}
    else:
        return {'passed': False, 'error_code': terminal_error_codes.get(status, '99')}


In [25]:
def validate_card_product(single_transaction, card_products_df):
    """
    Valida se o produto do cartão está ativo
    
    Args:
        single_transaction: Linha da transação com 'card_id', 'product_id' e 'amount'
        card_products_df: DataFrame dos produtos do cartão
    
    Returns:
        dict: {'passed': bool, 'error_code': str or None}
    """
    # Códigos de erro
    error_codes = {
        'inactive': '23',
    }
    
    card_id = single_transaction['card_id']
    product_id = single_transaction['product_id']
    
    # Buscar produto do cartão
    card_product = card_products_df[
        (card_products_df['card_id'] == card_id) & 
        (card_products_df['product_id'] == product_id)
    ]
    
    if card_product.empty:
        return {'passed': False, 'error_code': '99'}  # Produto do cartão não encontrado
    
    status = card_product['status'].iloc[0]
    
    if status != 'active':
        return {'passed': False, 'error_code': error_codes['inactive']}
    
    return {'passed': True, 'error_code': None}

In [26]:
def validate_product_active():
  """
  Valida se o produto como regra de negócio está ativo

  Returns:
      dict: {'passed': bool, 'error_code': str or None}
  """

  # Códigos de erro
  error_codes = {
      'inactive': '24',
  }

  # Para simulação, vamos assumir que todos os produtos estão ativos
  product_id = single_transaction['product_id']

  product = products_df[products_df['product_id'] == product_id]

  if product.empty or product['status'].iloc[0] != 'active':
      return {'passed': False, 'error_code': error_codes['inactive']}

  return {'passed': True, 'error_code': None}

In [27]:
def validate_modality_active(single_transaction, transaction_modalities_df, cards_df):
    """
    Valida se a modalidade do cartão está ativa
    
    Args:
        single_transaction: Linha da transação com 'card_id'
        transaction_modalities_df: DataFrame das modalidades de transação
        cards_df: DataFrame dos cartões
    
    Returns:
        dict: {'passed': bool, 'error_code': str or None}
    """
    # Códigos de erro
    error_codes = {
        'inactive': '25',
    }
    
    card_id = single_transaction['card_id']
    
    # Buscar cartão
    card = cards_df[cards_df['card_id'] == card_id]
    
    if card.empty:
        return {'passed': False, 'error_code': '99'}  # Cartão não encontrado
    
    modality_id = card['modality_id'].iloc[0]
    
    # Buscar modalidade
    modality = transaction_modalities_df[transaction_modalities_df['modality_id'] == modality_id]
    
    if modality.empty:
        return {'passed': False, 'error_code': '98'}  # Modalidade não encontrada
    
    # Para simulação, vamos assumir que todas as modalidades estão ativas
    return {'passed': True, 'error_code': None}

In [28]:
def validate_acquirer_active(single_transaction, terminals_df, acquirers_df):
    """
    Valida se a adquirente do terminal está ativa
    
    Args:
        single_transaction: Linha da transação com 'terminal_id'
        terminals_df: DataFrame dos terminais
        acquirers_df: DataFrame das adquirentes
    
    Returns:
        dict: {'passed': bool, 'error_code': str or None}
    """
    # Códigos de erro
    error_codes = {
        'inactive': '26',
    }
    
    terminal_id = single_transaction['terminal_id']
    
    # Buscar terminal
    terminal = terminals_df[terminals_df['terminal_id'] == terminal_id]
    
    if terminal.empty:
        return {'passed': False, 'error_code': '97'}  # Terminal não encontrado
    
    acquirer_id = terminal['acquirer_id'].iloc[0]
    
    # Buscar adquirente
    acquirer = acquirers_df[acquirers_df['acquirer_id'] == acquirer_id]
    
    if acquirer.empty:
        return {'passed': False, 'error_code': '96'}  # Adquirente não encontrada
    
    # Para simulação, vamos assumir que todas as adquirentes estão ativas
    return {'passed': True, 'error_code': None}

###  Temporal Validations

In [29]:
def validate_card_expiration(single_transaction, cards_df):
    """
    Valida se o cartão não está vencido
    """
    card_id = single_transaction['card_id']
    print(card_id)
    transaction_date = single_transaction['date']
    print(transaction_date)
    
    # Buscar cartão pelo ID
    card = cards_df[cards_df['card_id'] == card_id]
    print(card)
    
    if card.empty:
        return {'passed': False, 'error_code': '99'}  # Cartão não encontrado
    
    expiration_date = card['expiration_date'].iloc[0]
    
    # Verificar se cartão está vencido
    if transaction_date.date() > expiration_date:
        return {'passed': False, 'error_code': '54'}  # Cartão vencido
    
    return {'passed': True, 'error_code': None}

# Teste
test_result = validate_card_expiration(single_transaction, cards_df)

print(single_transaction)
print(f"Validação expiração: {test_result}")

1407
2025-08-31 21:12:52.726403
      card_id cardholder_id          card_number modality_id expiration_date  \
1407     1407     FUNC_1134  2385 4076 0569 3069     MOD_002      2025-11-29   

      cvv card_type              issuance_date   status  \
1407  416     debit 2025-09-21 17:11:46.472312  blocked   

                     created_at  
1407 2025-09-21 17:11:46.472315  
{'transaction_id': '33e925b0-7d08-4f29-846b-24dc9440ad01', 'card_id': np.int64(1407), 'product_id': 'PROD_VA', 'establishment_id': 'EST_000046', 'terminal_id': 'TERM_00000203', 'amount': 60.12, 'date': datetime.datetime(2025, 8, 31, 21, 12, 52, 726403), 'capture_type': 'contactless', 'response_code': None, 'response_message': None, 'created_at': Timestamp('2025-09-21 17:11:46.758001'), 'updated_at': Timestamp('2025-09-21 17:11:46.758013')}
Validação expiração: {'passed': True, 'error_code': None}


In [30]:
def cardholder_admission_validation(single_transaction, cards_df, cardholders_df):
    """
    Valida se a transação ocorreu após a admissão do portador
    """
    card_id = single_transaction['card_id']
    transaction_date = single_transaction['date']
    
    # Buscar cartão
    card = cards_df[cards_df['card_id'] == card_id]
    
    if card.empty:
        return {'passed': False, 'error_code': '99'}  # Cartão não encontrado
    
    cardholder_id = card['cardholder_id'].iloc[0]
    
    # Buscar portador
    cardholder = cardholders_df[cardholders_df['cardholder_id'] == cardholder_id]
    
    if cardholder.empty:
        return {'passed': False, 'error_code': '98'}  # Portador não encontrado
    
    admission_date = cardholder['admission_date'].iloc[0]
    
    # Verificar se transação é posterior à admissão
    if transaction_date.date() < admission_date:
        return {'passed': False, 'error_code': '56'}  # Transação antes da admissão
    
    return {'passed': True, 'error_code': None}

In [31]:
def establishment_accreditation_validation(single_transaction, establishments_df):
    """
    Valida se a transação ocorreu após o credenciamento do estabelecimento
    """
    establishment_id = single_transaction['establishment_id']
    transaction_date = single_transaction['date']
    
    # Buscar estabelecimento
    establishment = establishments_df[establishments_df['establishment_id'] == establishment_id]
    
    if establishment.empty:
        return {'passed': False, 'error_code': '97'}  # Estabelecimento não encontrado
    
    # Usar opening_date como accreditation_date (conforme estrutura atual)
    accreditation_date = establishment['opening_date'].iloc[0]
    
    # Verificar se transação é posterior ao credenciamento
    if transaction_date.date() < accreditation_date:
        return {'passed': False, 'error_code': '57'}  # Estabelecimento ainda não credenciado
    
    return {'passed': True, 'error_code': None}


In [32]:
def terminal_installation_validation(single_transaction, terminals_df):
    """
    Valida se a transação ocorreu após a instalação do terminal
    """
    terminal_id = single_transaction['terminal_id']
    transaction_date = single_transaction['date']
    
    # Buscar terminal
    terminal = terminals_df[terminals_df['terminal_id'] == terminal_id]
    
    if terminal.empty:
        return {'passed': False, 'error_code': '96'}  # Terminal não encontrado
    
    installation_date = terminal['installation_date'].iloc[0]
    
    # Verificar se transação é posterior à instalação
    if transaction_date.date() < installation_date:
        return {'passed': False, 'error_code': '58'}  # Terminal ainda não instalado
    
    return {'passed': True, 'error_code': None}

## VALIDAÇÕES DE REGRAS DE NEGÓCIO

### Gerar DataFrame de modalidades MCC (usando apenas MCCs dos estabelecimentos gerados)

In [ ]:
def structure_mcc_modalities_from_transaction(single_transaction, mcc_df, transaction_modalities_df, establishments_df, existing_mcc_modalities_df=None):
    """
    Alimenta a estrutura MCC x Modalidades baseado numa única transação
    Adiciona apenas o MCC do estabelecimento da transação
    """
    if existing_mcc_modalities_df is None:
        mcc_modalities = []
    else:
        mcc_modalities = existing_mcc_modalities_df.to_dict('records')
    
    # Pega o MCC do estabelecimento da transação
    establishment_id = single_transaction['establishment_id']
    establishment = establishments_df[establishments_df['establishment_id'] == establishment_id]
    
    if establishment.empty:
        return pd.DataFrame(mcc_modalities) if mcc_modalities else pd.DataFrame()
    
    mcc_code = establishment['mcc_code'].iloc[0]
    
    # Verifica se já existe esse MCC na estrutura
    if existing_mcc_modalities_df is not None:
        existing_mcc = existing_mcc_modalities_df[existing_mcc_modalities_df['mcc_code'] == mcc_code]
        if not existing_mcc.empty:
            return existing_mcc_modalities_df  # Já existe, retorna sem mudanças
    
    # Busca informações do MCC
    mcc_info = mcc_df[mcc_df['mcc_code'] == mcc_code]
    if mcc_info.empty:
        return pd.DataFrame(mcc_modalities) if mcc_modalities else pd.DataFrame()
    
    mcc_row = mcc_info.iloc[0]
    mcc_description = mcc_row['mcc_description']
    brands = mcc_row['valid_payment_brands']
    
    # Processa as modalidades válidas para este MCC
    if pd.notna(brands):
        for brand in brands.upper():
            modality_map = {'V': 'VISA', 'M': 'MASTERCARD', 'E': 'ELO'}
            if brand in modality_map:
                modality = transaction_modalities_df[
                    transaction_modalities_df['code'] == modality_map[brand]
                ]
                if not modality.empty:
                    mcc_modalities.append({
                        'mcc_code': mcc_code,
                        'modality_id': modality['modality_id'].iloc[0],
                        'mcc_description': mcc_description,
                        'observation': None,  # Por enquanto vazio
                        'active': True,
                        'created_at': pd.Timestamp.now()
                    })
    
    return pd.DataFrame(mcc_modalities)

## Business Rules Validations

### mcc_modality_mapping

In [ ]:



def validate_mcc_modality(single_transaction, mcc_modalities_df, establishments_df, cards_df):
    """Valida se MCC aceita modalidade do cartão da transação"""
    # Pega MCC do estabelecimento
    est = establishments_df[establishments_df['establishment_id'] == single_transaction['establishment_id']]
    if est.empty:
        return {'passed': False, 'error_code': '97'}
    
    mcc_code = est['mcc_code'].iloc[0]
    
    # Pega modalidade do cartão
    card = cards_df[cards_df['card_id'] == single_transaction['card_id']]
    if card.empty:
        return {'passed': False, 'error_code': '99'}
    
    modality_id = card['modality_id'].iloc[0]
    
    # Verifica se existe o mapeamento
    exists = mcc_modalities_df[
        (mcc_modalities_df['mcc_code'] == mcc_code) & 
        (mcc_modalities_df['modality_id'] == modality_id) &
        (mcc_modalities_df['active'] == True)
    ]
    
    return {'passed': not exists.empty, 'error_code': '59' if exists.empty else None}


# Exemplo de uso: alimentar estrutura com a transação atual
mcc_modalities_df = structure_mcc_modalities_from_transaction(
    single_transaction, mcc_df, transaction_modalities_df, establishments_df
)

# Validar a transação
result = validate_mcc_modality(single_transaction, mcc_modalities_df, establishments_df, cards_df)

In [38]:
def structure_cnae_products_from_transaction(single_transaction, cnaes_df, products_df, business_data, establishments_df, existing_cnae_products_df=None):
    """
    Alimenta a estrutura CNAE x Products baseado numa única transação
    Adiciona apenas o CNAE do estabelecimento da transação
    """
    if existing_cnae_products_df is None:
        cnae_products = []
    else:
        cnae_products = existing_cnae_products_df.to_dict('records')
    
    # Pega o CNAE do estabelecimento da transação
    establishment_id = single_transaction['establishment_id']
    establishment = establishments_df[establishments_df['establishment_id'] == establishment_id]
    
    if establishment.empty:
        return pd.DataFrame(cnae_products) if cnae_products else pd.DataFrame()
    
    cnae_code = establishment['main_cnae_id'].iloc[0]
    
    # Verifica se já existe esse CNAE na estrutura
    if existing_cnae_products_df is not None:
        existing_cnae = existing_cnae_products_df[existing_cnae_products_df['cnae_code'] == cnae_code]
        if not existing_cnae.empty:
            return existing_cnae_products_df  # Já existe, retorna sem mudanças
    
    # Busca informações do CNAE
    cnae_info = cnaes_df[cnaes_df['subclasse_id'] == cnae_code]
    if cnae_info.empty:
        return pd.DataFrame(cnae_products) if cnae_products else pd.DataFrame()
    
    cnae_row = cnae_info.iloc[0]
    cnae_description = cnae_row['subclasse']
    
    # Pega o mapeamento do business_rules.yaml
    cnae_products_mapping = business_data.get('cnae_products_mapping', {})
    
    # Verifica se existe mapeamento para este CNAE
    accepted_products = None
    
    # Procurar por correspondência exata primeiro
    if cnae_code in cnae_products_mapping:
        accepted_products = cnae_products_mapping[cnae_code]
    else:
        # Procurar por correspondência parcial (CNAE mais genérico)
        for mapped_cnae, products in cnae_products_mapping.items():
            if cnae_code.startswith(mapped_cnae):
                accepted_products = products
                break
    
    # Se encontrou produtos aceitos para este CNAE
    if accepted_products:
        for product_code in accepted_products:
            # Buscar o product_id correspondente no products_df
            product_match = products_df[products_df['code'] == product_code]
            
            if not product_match.empty:
                product_id = product_match['product_id'].iloc[0]
                
                cnae_products.append({
                    'cnae_code': cnae_code,
                    'product_id': product_id,
                    'cnae_description': cnae_description,
                    'observation': None,  # Por enquanto vazio
                    'active': True,
                    'created_at': pd.Timestamp.now()
                })
    
    return pd.DataFrame(cnae_products)


def validate_cnae_product(single_transaction, cnae_products_df, establishments_df):
    """Valida se CNAE aceita o produto da transação"""
    # Pega CNAE do estabelecimento
    est = establishments_df[establishments_df['establishment_id'] == single_transaction['establishment_id']]
    if est.empty:
        return {'passed': False, 'error_code': '97'}
    
    cnae_code = est['main_cnae_id'].iloc[0]
    product_id = single_transaction['product_id']
    
    # Verifica se existe o mapeamento
    exists = cnae_products_df[
        (cnae_products_df['cnae_code'] == cnae_code) & 
        (cnae_products_df['product_id'] == product_id) &
        (cnae_products_df['active'] == True)
    ]
    
    return {'passed': not exists.empty, 'error_code': '60' if exists.empty else None}


# Exemplo de uso: alimentar estrutura com a transação atual
cnae_products_df = structure_cnae_products_from_transaction(
    single_transaction, cnaes_df, products_df, business_data, establishments_df
)

# Validar a transação
result_cnae = validate_cnae_product(single_transaction, cnae_products_df, establishments_df)

print(f"CNAE x Product: {result_cnae}")
print(f"Registros na estrutura: {len(cnae_products_df)}")
print(f"Estrutura: {cnae_products_df.columns.tolist() if len(cnae_products_df) > 0 else 'DataFrame vazio'}")

CNAE x Product: {'passed': True, 'error_code': None}
Registros na estrutura: 1
Estrutura: ['cnae_code', 'product_id', 'cnae_description', 'observation', 'active', 'created_at']


In [ ]:
def validate_mcc_modality_rules_active(single_transaction, mcc_modalities_df, establishments_df, cards_df):
    """
    Valida se a regra MCC x Modalidade está ativa
    """
    # Pega MCC do estabelecimento
    est = establishments_df[establishments_df['establishment_id'] == single_transaction['establishment_id']]
    if est.empty:
        return {'passed': False, 'error_code': '97'}
    
    mcc_code = est['mcc_code'].iloc[0]
    
    # Pega modalidade do cartão
    card = cards_df[cards_df['card_id'] == single_transaction['card_id']]
    if card.empty:
        return {'passed': False, 'error_code': '99'}
    
    modality_id = card['modality_id'].iloc[0]
    
    # Verifica se existe o mapeamento e se está ativo
    rule = mcc_modalities_df[
        (mcc_modalities_df['mcc_code'] == mcc_code) & 
        (mcc_modalities_df['modality_id'] == modality_id)
    ]
    
    if rule.empty:
        return {'passed': False, 'error_code': '59'}  # Regra não existe
    
    # Verifica se a regra está ativa
    if not rule['active'].iloc[0]:
        return {'passed': False, 'error_code': '29'}  # Regra MCC desativada
    
    return {'passed': True, 'error_code': None}


def validate_cnae_product_rules_active(single_transaction, cnae_products_df, establishments_df):
    """
    Valida se a regra CNAE x Produto está ativa
    """
    # Pega CNAE do estabelecimento
    est = establishments_df[establishments_df['establishment_id'] == single_transaction['establishment_id']]
    if est.empty:
        return {'passed': False, 'error_code': '97'}
    
    cnae_code = est['main_cnae_id'].iloc[0]
    product_id = single_transaction['product_id']
    
    # Verifica se existe o mapeamento
    rule = cnae_products_df[
        (cnae_products_df['cnae_code'] == cnae_code) & 
        (cnae_products_df['product_id'] == product_id)
    ]
    
    if rule.empty:
        return {'passed': False, 'error_code': '60'}  # Regra não existe
    
    # Verifica se a regra está ativa
    if not rule['active'].iloc[0]:
        return {'passed': False, 'error_code': '30'}  # Regra CNAE desativada
    
    return {'passed': True, 'error_code': None}


# Teste das novas validações
result_mcc_active = validate_mcc_modality_rules_active(single_transaction, mcc_modalities_df, establishments_df, cards_df)
result_cnae_active = validate_cnae_product_rules_active(single_transaction, cnae_products_df, establishments_df)

print(f"MCC x Modalidade ativa: {result_mcc_active}")
print(f"CNAE x Produto ativo: {result_cnae_active}")

## VALIDAÇÕES DE SALDO E LIMITES

### balance_and_limits_validations

In [ ]:
def validate_current_balance(single_transaction, card_products_df):
    """
    Valida se há saldo suficiente no produto
    """
    card_id = single_transaction['card_id']
    product_id = single_transaction['product_id']
    amount = single_transaction['amount']
    
    # Buscar produto do cartão
    card_product = card_products_df[
        (card_products_df['card_id'] == card_id) & 
        (card_products_df['product_id'] == product_id)
    ]
    
    if card_product.empty:
        return {'passed': False, 'error_code': '99'}  # Produto do cartão não encontrado
    
    current_balance = card_product['current_balance'].iloc[0]
    
    # Verificar se há saldo suficiente
    if current_balance < amount:
        return {'passed': False, 'error_code': '51'}  # Saldo insuficiente
    
    return {'passed': True, 'error_code': None}

### daily_limit

In [ ]:
def validate_daily_limit(single_transaction, card_products_df, daily_usage=0):
    """
    Valida se o limite diário não foi ultrapassado
    
    Args:
        single_transaction: Transação atual
        card_products_df: DataFrame dos produtos do cartão
        daily_usage: Uso diário atual (simulado como 0 por padrão)
    """
    card_id = single_transaction['card_id']
    product_id = single_transaction['product_id']
    amount = single_transaction['amount']
    
    # Buscar produto do cartão
    card_product = card_products_df[
        (card_products_df['card_id'] == card_id) & 
        (card_products_df['product_id'] == product_id)
    ]
    
    if card_product.empty:
        return {'passed': False, 'error_code': '99'}  # Produto do cartão não encontrado
    
    daily_limit = card_product['daily_limit'].iloc[0]
    
    # Se limite = 0, não há restrição
    if daily_limit <= 0:
        return {'passed': True, 'error_code': None}
    
    # Verificar se limite diário seria ultrapassado
    if (daily_usage + amount) > daily_limit:
        return {'passed': False, 'error_code': '61'}  # Limite diário excedido
    
    return {'passed': True, 'error_code': None}

### monthly_limit

In [ ]:
def validate_monthly_limit(single_transaction, card_products_df, monthly_usage=0):
    """
    Valida se o limite mensal não foi ultrapassado
    
    Args:
        single_transaction: Transação atual
        card_products_df: DataFrame dos produtos do cartão
        monthly_usage: Uso mensal atual (simulado como 0 por padrão)
    """
    card_id = single_transaction['card_id']
    product_id = single_transaction['product_id']
    amount = single_transaction['amount']
    
    # Buscar produto do cartão
    card_product = card_products_df[
        (card_products_df['card_id'] == card_id) & 
        (card_products_df['product_id'] == product_id)
    ]
    
    if card_product.empty:
        return {'passed': False, 'error_code': '99'}  # Produto do cartão não encontrado
    
    monthly_limit = card_product['monthly_limit'].iloc[0]
    
    # Se limite = 0, não há restrição
    if monthly_limit <= 0:
        return {'passed': True, 'error_code': None}
    
    # Verificar se limite mensal seria ultrapassado
    if (monthly_usage + amount) > monthly_limit:
        return {'passed': False, 'error_code': '62'}  # Limite mensal excedido
    
    return {'passed': True, 'error_code': None}


# Teste das validações de saldo e limites
result_balance = validate_current_balance(single_transaction, card_products_df)
result_daily = validate_daily_limit(single_transaction, card_products_df, daily_usage=100)
result_monthly = validate_monthly_limit(single_transaction, card_products_df, monthly_usage=1000)

print(f"Saldo suficiente: {result_balance}")
print(f"Limite diário: {result_daily}")
print(f"Limite mensal: {result_monthly}")

## VALIDAÇÕES DE INFRAESTRUTURA

### terminal_acquirer_match

In [ ]:
def validate_terminal_acquirer_match(single_transaction, terminals_df, cards_df, transaction_modalities_df):
    """
    Valida se o terminal pertence ao adquirente compatível com a modalidade do cartão
    """
    terminal_id = single_transaction['terminal_id']
    card_id = single_transaction['card_id']
    
    # Buscar terminal
    terminal = terminals_df[terminals_df['terminal_id'] == terminal_id]
    if terminal.empty:
        return {'passed': False, 'error_code': '96'}  # Terminal não encontrado
    
    terminal_acquirer_id = terminal['acquirer_id'].iloc[0]
    
    # Buscar cartão para pegar modalidade
    card = cards_df[cards_df['card_id'] == card_id]
    if card.empty:
        return {'passed': False, 'error_code': '99'}  # Cartão não encontrado
    
    modality_id = card['modality_id'].iloc[0]
    
    # Buscar modalidade para verificar se é compatível
    modality = transaction_modalities_df[transaction_modalities_df['modality_id'] == modality_id]
    if modality.empty:
        return {'passed': False, 'error_code': '98'}  # Modalidade não encontrada
    
    # Para simplificação, vamos assumir que todos os terminais são compatíveis
    # Em um cenário real, teria uma tabela de compatibilidade terminal x modalidade
    return {'passed': True, 'error_code': None}

### capture_type_support

In [ ]:
def validate_capture_type_support(single_transaction, terminals_df, transaction_modalities_df, cards_df):
    """
    Valida se o tipo de captura é suportado pelo terminal e modalidade
    """
    terminal_id = single_transaction['terminal_id']
    capture_type = single_transaction['capture_type']
    card_id = single_transaction['card_id']
    
    # Buscar terminal
    terminal = terminals_df[terminals_df['terminal_id'] == terminal_id]
    if terminal.empty:
        return {'passed': False, 'error_code': '96'}  # Terminal não encontrado
    
    terminal_type = terminal['terminal_type'].iloc[0]
    
    # Buscar cartão para pegar modalidade
    card = cards_df[cards_df['card_id'] == card_id]
    if card.empty:
        return {'passed': False, 'error_code': '99'}  # Cartão não encontrado
    
    modality_id = card['modality_id'].iloc[0]
    
    # Buscar modalidade
    modality = transaction_modalities_df[transaction_modalities_df['modality_id'] == modality_id]
    if modality.empty:
        return {'passed': False, 'error_code': '98'}  # Modalidade não encontrada
    
    # Regras de compatibilidade simplificadas
    # POS suporta: chip, magnetic_stripe, contactless
    # TEF suporta: chip, magnetic_stripe
    # Modalidades digitais (NFC, Apple Pay, etc.) só funcionam com POS
    
    if terminal_type == 'POS':
        supported_captures = ['chip', 'magnetic_stripe', 'contactless', 'nfc']
    elif terminal_type == 'TEF':
        supported_captures = ['chip', 'magnetic_stripe']
    else:
        supported_captures = ['chip', 'magnetic_stripe']  # Default
    
    if capture_type not in supported_captures:
        return {'passed': False, 'error_code': '32'}  # Tipo de captura não suportado
    
    return {'passed': True, 'error_code': None}


# Teste das validações de infraestrutura
result_terminal_acquirer = validate_terminal_acquirer_match(single_transaction, terminals_df, cards_df, transaction_modalities_df)
result_capture_support = validate_capture_type_support(single_transaction, terminals_df, transaction_modalities_df, cards_df)

print(f"Terminal/Adquirente compatível: {result_terminal_acquirer}")
print(f"Tipo de captura suportado: {result_capture_support}")
print(f"Transação usando capture_type: {single_transaction['capture_type']}")

### Autorizador de transações

In [ ]:
def generate_authorizations(transactions_df):
    """
    Autorizador simples que processa transações e gera autorizações
    Simula aprovações/negações baseado em regras simples
    """
    authorizations = []
    
    for _, transaction in transactions_df.iterrows():
        transaction_information = {
            "transaction_id": transaction["transaction_id"],
            "amount": transaction["amount"],
            "currency": transaction["currency"],
            "date": transaction["date"]
        }

        def authorization_logic():
            # 85% de aprovação, 15% de negação
            is_approved = random.random() < 0.85

        def validation_establishment():
            if random.random() < 0.05:
                return "12", "Estabelecimento inválido"
            return None, None

        authorization = {
            "authorization_id": fake.uuid4(),
            "transaction_id": transaction["transaction_id"],
            "authorization_code": auth_code,
            "nsu": f"NSU{random.randint(100000000, 999999999)}",
            "authorization_datetime": transaction["date"],
            "status": status,
            "response_code": response_code,
            "response_message": response_message,
            "created_at": pd.Timestamp.now()
        }
        authorizations.append(authorization)
    
    return pd.DataFrame(authorizations)

def update_transactions_with_authorization(transactions_df, authorizations_df):
    """
    Atualiza as transações com os códigos de resposta do autorizador
    """
    # Criar um merge baseado no transaction_id
    updated_transactions = transactions_df.merge(
        authorizations_df[['transaction_id', 'response_code', 'response_message']], 
        on='transaction_id', 
        how='left'
    )
    
    # Atualizar os campos que estavam None
    updated_transactions['response_code'] = updated_transactions['response_code_y']
    updated_transactions['response_message'] = updated_transactions['response_message_y']
    updated_transactions['updated_at'] = pd.Timestamp.now()
    
    # Remover colunas duplicadas do merge
    updated_transactions = updated_transactions.drop(columns=['response_code_x', 'response_code_y', 
                                                              'response_message_x', 'response_message_y'])
    
    return updated_transactions

# Processar transações
authorizations_df = generate_authorizations(transactions_df)
transactions_df = update_transactions_with_authorization(transactions_df, authorizations_df)